In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1. CARREGAR E INSPECIONAR NDBI
# ============================================================
df_ndbi = pd.read_csv('../data/raw/gee/ndbi_2018_2024.csv')

print(f"NDBI carregado: {df_ndbi.shape}")
print(f"Colunas: {df_ndbi.columns.tolist()}")
print(f"\nAmostra:")
print(df_ndbi.head(8).to_string(index=False))
print(f"\nNulos:")
print(df_ndbi.isnull().sum())
print(f"\nEstatísticas NDBI:")
print(df_ndbi['ndbi'].describe())

NDBI carregado: (84, 8)
Colunas: ['system:index', 'ano', 'data', 'mes', 'n_imagens', 'ndbi', 'ndvi', '.geo']

Amostra:
 system:index    ano    data  mes  n_imagens      ndbi     ndvi                                   .geo
            0 2018.0 2018-01  1.0          0       NaN      NaN {"type":"MultiPoint","coordinates":[]}
            1 2018.0 2018-02  2.0          0       NaN      NaN {"type":"MultiPoint","coordinates":[]}
            2 2018.0 2018-03  3.0          0       NaN      NaN {"type":"MultiPoint","coordinates":[]}
            3 2018.0 2018-04  4.0          0       NaN      NaN {"type":"MultiPoint","coordinates":[]}
            4 2018.0 2018-05  5.0          4 -0.038049 0.607621 {"type":"MultiPoint","coordinates":[]}
            5 2018.0 2018-06  6.0          1 -0.072640 0.655926 {"type":"MultiPoint","coordinates":[]}
            6 2018.0 2018-07  7.0          5  0.076089 0.497381 {"type":"MultiPoint","coordinates":[]}
            7 2018.0 2018-08  8.0          0       NaN   

In [2]:
# ============================================================
# 2. LIMPAR + IMPUTAR + INTEGRAR AO DATASET_FEATURES_V3_TRENDS
# ============================================================

# Limpar colunas desnecessárias
df_ndbi_clean = df_ndbi[['data', 'ano', 'mes', 'ndbi', 'ndvi', 'n_imagens']].copy()
df_ndbi_clean['data'] = pd.to_datetime(df_ndbi_clean['data'])

# Imputar nulos com interpolação sazonal (mesma abordagem do NDVI/NDWI)
df_ndbi_clean = df_ndbi_clean.sort_values('data').reset_index(drop=True)
df_ndbi_clean['ndbi'] = df_ndbi_clean['ndbi'].interpolate(method='linear')
df_ndbi_clean['ndvi'] = df_ndbi_clean['ndvi'].interpolate(method='linear')

# Preencher bordas se ainda houver nulos
df_ndbi_clean['ndbi'] = df_ndbi_clean['ndbi'].ffill().bfill()
df_ndbi_clean['ndvi'] = df_ndbi_clean['ndvi'].ffill().bfill()

print(f"Nulos após imputação: {df_ndbi_clean[['ndbi','ndvi']].isnull().sum().sum()}")
print(f"Registros: {len(df_ndbi_clean)}")

# Carregar dataset v3_trends
df_v3t = pd.read_parquet('../data/gold/dataset_features_v3_trends.parquet')
df_v3t['data'] = pd.to_datetime(df_v3t['data'])

# Expandir NDBI mensal → diário (ffill por mês)
df_ndbi_diario = (
    df_ndbi_clean[['data', 'ndbi']]
    .set_index('data')
    .resample('D')
    .ffill()
    .reset_index()
)
df_ndbi_diario.columns = ['data', 'ndbi_gee']

# Merge
df_v4 = df_v3t.merge(df_ndbi_diario, on='data', how='left')
df_v4['ndbi_gee'] = df_v4['ndbi_gee'].ffill().bfill()

# Lags do NDBI (mensal — lags de 30 e 60 dias)
df_v4 = df_v4.sort_values('data').reset_index(drop=True)
df_v4['ndbi_lag_30d'] = df_v4['ndbi_gee'].shift(30)
df_v4['ndbi_lag_60d'] = df_v4['ndbi_gee'].shift(60)

print(f"\ndataset_features_v4: {df_v4.shape}")
print(f"Novas features: ndbi_gee, ndbi_lag_30d, ndbi_lag_60d")
print(f"Nulos ndbi_gee: {df_v4['ndbi_gee'].isnull().sum()}")

# Correlação NDBI × casos
corr = df_v4['ndbi_gee'].corr(df_v4['casos'])
print(f"\nCorrelação NDBI × casos: r={corr:.3f}")

# Salvar
df_v4.to_parquet('../data/gold/dataset_features_v4.parquet', index=False)
print(f"\nSalvo: data/gold/dataset_features_v4.parquet")
print(f"   {df_v4.shape[1]} features no total")

Nulos após imputação: 0
Registros: 84

dataset_features_v4: (2242, 67)
Novas features: ndbi_gee, ndbi_lag_30d, ndbi_lag_60d
Nulos ndbi_gee: 0

Correlação NDBI × casos: r=-0.446

Salvo: data/gold/dataset_features_v4.parquet
   67 features no total
